
# 01 OpenCV Fundamentals

OpenCV is the library you'll reach for most often in document forensics work. This notebook covers the
core operations everything else builds on: loading images as arrays, color spaces, filtering, edge
detection, and contour detection ending with a real, practical task: finding a document's boundary
within a photo, the first step before any forensic check can run.

## The first gotcha: OpenCV uses BGR, not RGB

Every other Python imaging tool (Pillow, matplotlib) expects color images as **RGB** (red, green, blue
channel order). OpenCV, for historical reasons, uses **BGR**. This is the single most common source of
"why do my colors look wrong" bugs for beginners plot an OpenCV image directly with matplotlib without
converting, and reds and blues swap.


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# Generate a synthetic "scanned document" photo a white rectangle (the document) at a
# slight angle, on a gray "desk" background with some noise, standing in for a real scan
np.random.seed(0)
canvas = Image.new("RGB", (500, 400), (90, 90, 95))  # gray background = the "desk"
draw = ImageDraw.Draw(canvas)

# The "document" a white rectangle with a few dark lines standing in for text
doc_corners = [(80, 60), (420, 40), (440, 340), (60, 360)]  # slightly skewed quadrilateral
draw.polygon(doc_corners, fill=(250, 250, 248))
for y in range(90, 320, 25):
    draw.line([(100, y), (400, y)], fill=(40, 40, 40), width=3)

document_photo = np.array(canvas)  # this is RGB, since PIL produced it

# Add a little sensor noise, like a real photographed scan would have
noise = np.random.normal(0, 8, document_photo.shape).astype(np.int16)
document_photo = np.clip(document_photo.astype(np.int16) + noise, 0, 255).astype(np.uint8)

plt.figure(figsize=(6, 5))
plt.imshow(document_photo)
plt.title("Synthetic scanned document photo (RGB)")
plt.axis("off")
plt.show()



## Loading images the OpenCV way

`cv2.imread()` reads a file directly into BGR order. Since we generated our image with PIL (RGB), we
need to explicitly convert this cell shows both the "loading from disk" pattern and the conversion.


In [ ]:

# Save what we generated, then reload it the way you would with a real scanned file
cv2.imwrite("/tmp/document_photo.png", cv2.cvtColor(document_photo, cv2.COLOR_RGB2BGR))

img_bgr = cv2.imread("./images/deed.jpg")  # OpenCV always loads as BGR
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)  # convert for correct display/comparison with PIL-based tools

print("Shape:", img_bgr.shape, "(height, width, channels)")
print("dtype:", img_bgr.dtype)



## Grayscale, blurring, and thresholding

Most forensic and boundary-detection operations work on grayscale images color is usually noise for
these purposes, not signal.


In [ ]:

gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# A slight blur reduces noise before edge detection edge detectors are very sensitive to
# pixel-level noise, so this step matters more than it looks like it should
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(gray, cmap="gray"); axes[0].set_title("Grayscale"); axes[0].axis("off")
axes[1].imshow(blurred, cmap="gray"); axes[1].set_title("Blurred"); axes[1].axis("off")
plt.show()



## Edge detection with Canny

The Canny edge detector finds sharp intensity transitions exactly the kind of transition you'd see at
the boundary between the document and the desk behind it.


In [ ]:

edges = cv2.Canny(blurred, threshold1=50, threshold2=150)

plt.figure(figsize=(6, 5))
plt.imshow(edges, cmap="gray")
plt.title("Canny edges")
plt.axis("off")
plt.show()



## Contour detection finding the document's actual boundary

A contour is a connected curve of points along a boundary. `cv2.findContours()` finds all of them;
you then filter for the one that's actually the document typically the largest, roughly
quadrilateral one.


In [ ]:

contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Sort by area, largest first the document should be the biggest contiguous shape in the photo
contours = sorted(contours, key=cv2.contourArea, reverse=True)
largest = contours[0]

# Approximate the contour as a polygon with fewer points useful to confirm it's roughly
# quadrilateral (4 corners), which a real document boundary should be
perimeter = cv2.arcLength(largest, True)
approx = cv2.approxPolyDP(largest, 0.02 * perimeter, True)

print(f"Found {len(contours)} contours.")
print(f"Largest contour approximates to a {len(approx)}-sided shape.")

result = img_rgb.copy()
cv2.drawContours(result, [approx], -1, (255, 0, 0), 3)

plt.figure(figsize=(6, 5))
plt.imshow(result)
plt.title("Detected document boundary")
plt.axis("off")
plt.show()



This detected boundary is exactly what you'd use to **crop and deskew** a photographed document before
running any forensic analysis on it forensic checks (notebook 02) need a clean, cropped, upright image
to give meaningful results, not a photo of a document sitting at an angle on a desk.

## Exercises

### Exercise 1
Generate a second synthetic document photo with a different skew angle and background color, and
confirm the contour-detection pipeline above still correctly finds a 4-sided boundary.


In [ ]:
# Your code here


#### Solution

In [ ]:

canvas2 = Image.new("RGB", (500, 400), (60, 70, 60))
draw2 = ImageDraw.Draw(canvas2)
doc_corners2 = [(50, 90), (430, 50), (460, 330), (90, 370)]
draw2.polygon(doc_corners2, fill=(245, 245, 240))
for y in range(120, 320, 25):
    draw2.line([(110, y), (410, y)], fill=(30, 30, 30), width=3)

img2 = np.array(canvas2)
gray2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
blurred2 = cv2.GaussianBlur(gray2, (5, 5), 0)
edges2 = cv2.Canny(blurred2, 50, 150)
contours2, _ = cv2.findContours(edges2, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours2 = sorted(contours2, key=cv2.contourArea, reverse=True)
perimeter2 = cv2.arcLength(contours2[0], True)
approx2 = cv2.approxPolyDP(contours2[0], 0.02 * perimeter2, True)
print(f"Detected a {len(approx2)}-sided boundary.")



### Exercise 2
Write a function `detect_document_boundary(image_rgb)` that wraps the full pipeline above (grayscale →
blur → Canny → contours → approxPolyDP) and returns the 4-point boundary as a NumPy array, ready to
reuse in later notebooks.


In [ ]:
# Your code here


#### Solution

In [ ]:

def detect_document_boundary(image_rgb):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    perimeter = cv2.arcLength(contours[0], True)
    approx = cv2.approxPolyDP(contours[0], 0.02 * perimeter, True)
    return approx.reshape(-1, 2)

boundary = detect_document_boundary(document_photo)
print("Boundary points:\n", boundary)



## What's next

You can now find a document within a photo. The next question is the actual forensic one: **is this
specific document genuine, or has it been tampered with?** `02_document_forensics.ipynb` covers three
concrete, practical techniques for that.
